# Pronóstico de fallas por análisis de aceite — **BayesLSTM-VAE**
### Flota diésel minera (QSK78 / QSK95) · Notebook ejecutable en Google Colab / Jupyter

Este notebook implementa, **de principio a fin y con explicaciones**, un modelo predictivo
híbrido que combina:

1. **Bayesian LSTM** — predice la tendencia futura de los parámetros del aceite y **cuantifica la incertidumbre**. Se ofrecen dos variantes:
   - **MC Dropout** (rápido) y
   - **Bayes by Backprop / BBB** (fiel al paper).
2. **Matriz de firmas metálicas (F)** — traduce los metales en **modos de falla** físicos (cojinetes, cilindro, aire, refrigerante, combustión).
3. **VAE** — detecta **anomalías** por error de reconstrucción.
4. **Score de riesgo multinivel (0–3)** — combina anomalía + tendencia + incertidumbre por modo.

> **Referencia:** Chen, Y. et al. (2026). *Diesel engine lubricating oil fault prognosis: A hybrid Bayesian LSTM and deep generative model architecture for multilayer anomaly detection.* **Tribology International, 215**, 111434.

**Cómo usarlo:** ejecuta las celdas en orden. Por defecto corre con **datos sintéticos** (no necesita base de datos), así que funciona en cualquier Colab/Jupyter. Más abajo se explica cómo conectar tu **Azure SQL** real de forma segura.

## 0. Instalación de dependencias
En Colab esto instala todo lo necesario. En un Jupyter local, normalmente ya tendrás varias de estas librerías.

In [ ]:
# En Colab/Jupyter:
!pip install -q numpy pandas scikit-learn torch matplotlib sqlalchemy python-dotenv 2>/dev/null
print("Dependencias listas")

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as Fnn
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DEVICE)

## 1. Configuración del modelo
Todo lo ajustable vive en este diccionario: variables del aceite, contexto, modos de falla,
la matriz de firmas **F**, hiperparámetros y los pesos del score de riesgo.

> El **orden** de `OIL_VARS` define el orden de columnas de `x` y de las columnas de `F`.

In [ ]:
CFG = {
    # 15 parámetros del aceite (vector x)
    "oil_vars": ["Fe","Cu","Pb","Sn","Al","Cr","Ni","Si","Na","K",
                 "Ox","Nit","Hollin","V100","TBN"],
    # contexto operativo disponible en [Oil].[LaboratoryData]
    "context_vars": ["HorasAceite","HorasComp","Horometro"],
    # modos de falla (filas de F)
    "failure_modes": ["Cojinetes","Cilindro","Aire","Refrigerante","Combustion"],

    # Matriz de firmas F (k x d), columnas en el orden de oil_vars.
    # +1 = el parámetro sube ante la falla; -1 = baja (TBN cae con refrigerante).
    "signature_matrix": {
        #               Fe Cu Pb Sn Al Cr Ni Si Na  K Ox Nit Hol V10 TBN
        "Cojinetes":    [0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,  0],
        "Cilindro":     [1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0,  0],
        "Aire":         [1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0,  0],
        "Refrigerante": [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, -1],
        "Combustion":   [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0,  0],
    },

    # ---- Modelo ----
    "method": "mc_dropout",   # "mc_dropout" (rápido) o "bbb" (Bayes by Backprop)
    "window_size": 10,        # T
    "horizon": 1,             # H
    "lstm_hidden": 64,
    "mc_dropout": 0.3,
    "mc_samples": 30,
    "bbb_prior_sigma": 0.1,
    "bbb_kl_weight": 0.001,
    "vae_latent": 8,
    "vae_hidden": 64,

    # ---- Entrenamiento ----
    "epochs_lstm": 40,
    "epochs_vae": 60,
    "batch_size": 64,
    "lr": 1e-3,

    # ---- Score de riesgo ----
    "alpha": 1.0, "beta": 1.0, "gamma": 0.5,   # pesos
}
D = len(CFG["oil_vars"]); K = len(CFG["failure_modes"])
print(f"d (parámetros)={D} | k (modos)={K} | contexto={len(CFG['context_vars'])}")

## 2. Datos

### Opción A — Datos sintéticos (por defecto)
Genera una flota de 20 motores sanos + 5 con falla progresiva (uno por modo) para validar
todo el pipeline sin base de datos.

### Opción B — Tu Azure SQL real (`[Oil].[LaboratoryData]`)
Se incluye la celda de conexión **segura** más abajo. Nunca escribas la contraseña en el
notebook: se pide con `getpass` en memoria.

El mapeo de columnas ya corresponde a tu esquema:
`Fe→Fe_ppm`, `Ox→Oxidacion`, `Nit→Nitracion`, `HorasAceite→HorasDeAceite`, etc. La serie de
cada motor se agrupa por `ComponentSerialNumber` y se filtra `Compartimiento` a motor.

In [ ]:
# ---------- Generador sintético ----------
_BASE = {"Fe":15,"Cu":4,"Pb":3,"Sn":1.5,"Al":3,"Cr":1.5,"Ni":0.8,"Si":8,"Na":3,"K":2,
         "Ox":10,"Nit":8,"Hollin":0.2,"V100":14.5,"TBN":9.0}
_DRIVERS = {
    "Cojinetes":{"Cu":6,"Pb":5,"Sn":3}, "Cilindro":{"Fe":10,"Cr":2.5,"Al":4},
    "Aire":{"Si":12,"Al":5,"Fe":6}, "Refrigerante":{"Na":8,"K":6,"TBN":-4},
    "Combustion":{"Ox":15,"Nit":12,"Hollin":1.5},
}
def _engine(eid, n, rng, mode=None, onset=0.5):
    rows=[]; horas=0.0; hcomp=rng.uniform(2000,8000)
    for t in range(n):
        horas+=rng.uniform(180,260); hcomp+=horas
        rec={v:max(0.0,_BASE[v]+rng.normal(0,_BASE[v]*0.08)) for v in CFG["oil_vars"]}
        if mode:
            frac=t/(n-1)
            if frac>=onset:
                sev=((frac-onset)/(1-onset))**1.6
                for m,g in _DRIVERS[mode].items():
                    rec[m]=max(0.0, rec[m]+g*sev*(3+2*frac))
        rec.update({"equipo":eid,
                    "fecha_muestra":pd.Timestamp("2024-01-01")+pd.Timedelta(days=14*t),
                    "HorasAceite":horas,"HorasComp":hcomp,"Horometro":hcomp+rng.uniform(0,500),
                    "_modo_real":mode or "Sano"})
        rows.append(rec)
    return rows
def generate_fleet(n_healthy=20, n_faulty=5, n=40, seed=SEED):
    rng=np.random.default_rng(seed); rows=[]
    for i in range(n_healthy): rows+=_engine(f"T3{100+i:03d}", n, rng)
    modes=list(_DRIVERS)
    for i in range(n_faulty):
        rows+=_engine(f"T3{170+i:03d}", n, rng, mode=modes[i%len(modes)], onset=rng.uniform(0.4,0.6))
    cols=["equipo","fecha_muestra"]+CFG["oil_vars"]+CFG["context_vars"]+["_modo_real"]
    return pd.DataFrame(rows)[cols].sort_values(["equipo","fecha_muestra"]).reset_index(drop=True)

df = generate_fleet()
print(df.shape, "| motores:", df["equipo"].nunique())
df.head()

In [ ]:
# ---------- Opción B: conexión SEGURA a Azure SQL (ejecutar solo si la usarás) ----------
USAR_SQL = False   # cámbialo a True para leer de tu base real

if USAR_SQL:
    from getpass import getpass
    from sqlalchemy import create_engine, text
    import urllib.parse as up

    # En Colab necesitas el driver ODBC (descomenta):
    # !curl https://packages.microsoft.com/keys/microsoft.asc | apt-key add -
    # !curl https://packages.microsoft.com/config/ubuntu/$(lsb_release -rs)/prod.list > /etc/apt/sources.list.d/mssql-release.list
    # !ACCEPT_EULA=Y apt-get install -y msodbcsql18

    user = "usuario_lectura_sql"
    pwd  = getpass("Contraseña de lectura (no se guarda): ")   # NUNCA hardcodear
    server = "serverbd-osconfiabilidad.database.windows.net"
    dbname = "bd_kmmp_osconfiabilidad"
    url = (f"mssql+pyodbc://{user}:{up.quote_plus(pwd)}@{server}:1433/{dbname}"
           "?driver=ODBC+Driver+18+for+SQL+Server&Encrypt=yes&TrustServerCertificate=no")
    engine = create_engine(url, pool_pre_ping=True)

    cmap = {"Fe":"Fe_ppm","Cu":"Cu_ppm","Pb":"Pb_ppm","Sn":"Sn_ppm","Al":"Al_ppm",
            "Cr":"Cr_ppm","Ni":"Ni_ppm","Si":"Si_ppm","Na":"Na_ppm","K":"K_ppm",
            "Ox":"Oxidacion","Nit":"Nitracion","Hollin":"Hollin","V100":"V100","TBN":"TBN",
            "HorasAceite":"HorasDeAceite","HorasComp":"HorasDelComp","Horometro":"Horometro"}
    sel = ["[ComponentSerialNumber] AS equipo","[FechaMuestreo] AS fecha_muestra",
           "[Estado] AS estado_bd"]
    sel += [f"[{cmap[c]}] AS {c}" for c in CFG["oil_vars"]+CFG["context_vars"]]
    q = ("SELECT " + ", ".join(sel) + " FROM [Oil].[LaboratoryData] "
         "WHERE [Compartimiento] IN ('MOTOR','Engine','MOTOR DIESEL')")
    with engine.connect() as con:
        df = pd.read_sql(text(q), con)
    df["fecha_muestra"] = pd.to_datetime(df["fecha_muestra"])
    df["_modo_real"] = df["estado_bd"].apply(
        lambda s: "Sano" if str(s).upper()=="NORMAL" else "Anomalo")
    df = df.dropna(subset=["equipo","fecha_muestra"]).sort_values(["equipo","fecha_muestra"])
    print("Filas leídas de Azure:", len(df))


## 3. Matriz de firmas metálicas $F$
La transformación $m_t = F\,x_t$ convierte el vector de metales en **activaciones por modo de falla**.
Es la capa de conocimiento tribológico que hace al modelo **explicable**.

In [ ]:
F = torch.tensor([CFG["signature_matrix"][m] for m in CFG["failure_modes"]],
                 dtype=torch.float32)
print("F:", tuple(F.shape), "(k x d)")
pd.DataFrame(F.numpy(), index=CFG["failure_modes"], columns=CFG["oil_vars"]).astype(int)

## 4. Escalado y ventanas temporales
El LSTM necesita secuencias. Por cada motor construimos ventanas deslizantes de longitud $T$:
la entrada es $[u_{t-T+1},\dots,u_t]$ y el objetivo es $x_{t+H}$ (los metales a horizonte $H$).

In [ ]:
from sklearn.preprocessing import StandardScaler

FEAT = CFG["oil_vars"] + CFG["context_vars"]
scaler = StandardScaler().fit(df[FEAT].astype(float).values)

def make_windows(df, T, H):
    X,Y,meta=[],[],[]
    for eid,g in df.groupby("equipo"):
        g=g.sort_values("fecha_muestra").reset_index(drop=True)
        if len(g)<T+H: continue
        feats=scaler.transform(g[FEAT].astype(float).values)
        for t in range(T-1, len(g)-H):
            X.append(feats[t-T+1:t+1,:]); Y.append(feats[t+H,:D])
            r=g.iloc[t+H]
            meta.append({"equipo":eid,"fecha_muestra":r["fecha_muestra"],
                         "modo_real":r.get("_modo_real","NA")})
    return (np.asarray(X,dtype=np.float32), np.asarray(Y,dtype=np.float32),
            pd.DataFrame(meta))

X,Y,meta = make_windows(df, CFG["window_size"], CFG["horizon"])
print("X:",X.shape," Y:",Y.shape," ventanas:",len(meta))

## 5. Bayesian LSTM

### 5.1. MC Dropout (rápido)
Se deja el *dropout* activo también en inferencia y se hacen $N$ pasadas estocásticas:
la **media** es la predicción y la **varianza** estima la incertidumbre (Gal & Ghahramani, 2016).

### 5.2. Bayes by Backprop — BBB (fiel al paper)
Cada peso es una variable aleatoria $q(W)=\mathcal{N}(\mu,\sigma^2)$ con $\sigma=\text{softplus}(\rho)$.
En cada *forward* se **muestrea** $W=\mu+\sigma\,\epsilon$ (reparametrización) y se minimiza la
**energía libre variacional**:
$$\mathcal{L} = \underbrace{\mathbb{E}_q[-\log p(D\mid W)]}_{\text{error de predicción}} + \underbrace{\mathrm{KL}[q(W)\,\|\,p(W)]}_{\text{regularización al prior}}$$
La incertidumbre sale de muestrear pesos varias veces en inferencia.

In [ ]:
# ---- 5.1 MC Dropout ----
class LSTM_MC(nn.Module):
    def __init__(self, in_dim, hid, out_dim, dropout=0.3):
        super().__init__()
        self.lstm=nn.LSTM(in_dim,hid,2,batch_first=True,dropout=dropout)
        self.drop=nn.Dropout(dropout); self.fc=nn.Linear(hid,out_dim)
    def forward(self,x):
        o,_=self.lstm(x); return self.fc(self.drop(o[:,-1,:]))
    @torch.no_grad()
    def predict_unc(self,x,n=30):
        self.train()
        p=np.stack([self.forward(x).cpu().numpy() for _ in range(n)],0)
        return p.mean(0), p.var(0)

# ---- 5.2 Bayes by Backprop ----
class BayesLinear(nn.Module):
    def __init__(self, i, o, prior=0.1):
        super().__init__(); self.prior=prior
        self.w_mu=nn.Parameter(torch.empty(o,i)); self.w_rho=nn.Parameter(torch.empty(o,i))
        self.b_mu=nn.Parameter(torch.zeros(o));   self.b_rho=nn.Parameter(torch.full((o,),-5.0))
        nn.init.kaiming_uniform_(self.w_mu,a=5**0.5); nn.init.constant_(self.w_rho,-5.0)
        self.kl=0.0
    @staticmethod
    def _kl(mu,sig,prior):
        return torch.sum(np.log(prior)-torch.log(sig)+(sig**2+mu**2)/(2*prior**2)-0.5)
    def forward(self,x):
        ws=Fnn.softplus(self.w_rho); bs=Fnn.softplus(self.b_rho)
        w=self.w_mu+ws*torch.randn_like(ws); b=self.b_mu+bs*torch.randn_like(bs)
        self.kl=self._kl(self.w_mu,ws,self.prior)+self._kl(self.b_mu,bs,self.prior)
        return Fnn.linear(x,w,b)

class BayesLSTMCell(nn.Module):
    def __init__(self, in_dim, hid, prior=0.1):
        super().__init__(); self.hid=hid
        self.x2h=BayesLinear(in_dim,4*hid,prior); self.h2h=BayesLinear(hid,4*hid,prior)
    def forward(self,x,state):
        h,c=state; g=self.x2h(x)+self.h2h(h)
        i,f,gg,o=g.chunk(4,1)
        i,f,o=torch.sigmoid(i),torch.sigmoid(f),torch.sigmoid(o); gg=torch.tanh(gg)
        c=f*c+i*gg; h=o*torch.tanh(c); return h,c
    @property
    def kl(self): return self.x2h.kl+self.h2h.kl

class LSTM_BBB(nn.Module):
    def __init__(self, in_dim, hid, out_dim, prior=0.1):
        super().__init__(); self.hid=hid
        self.cell=BayesLSTMCell(in_dim,hid,prior); self.head=BayesLinear(hid,out_dim,prior)
    def forward(self,x):
        B,T,_=x.shape; h=x.new_zeros(B,self.hid); c=x.new_zeros(B,self.hid)
        for t in range(T): h,c=self.cell(x[:,t,:],(h,c))
        return self.head(h)
    def kl_loss(self): return self.cell.kl+self.head.kl
    @torch.no_grad()
    def predict_unc(self,x,n=30):
        self.eval()
        p=np.stack([self.forward(x).cpu().numpy() for _ in range(n)],0)
        return p.mean(0), p.var(0)
print("Modelos definidos.")

### 5.3. Entrenamiento del LSTM
Selecciona el método con `CFG["method"]` (`"mc_dropout"` o `"bbb"`).

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
Xt=torch.tensor(X); Yt=torch.tensor(Y)
dl=DataLoader(TensorDataset(Xt,Yt), batch_size=CFG["batch_size"], shuffle=True)

if CFG["method"]=="bbb":
    lstm=LSTM_BBB(X.shape[2], CFG["lstm_hidden"], D, CFG["bbb_prior_sigma"]).to(DEVICE)
    opt=torch.optim.Adam(lstm.parameters(), lr=CFG["lr"])
    klw=CFG["bbb_kl_weight"]/max(1,len(dl))
    for ep in range(CFG["epochs_lstm"]):
        lstm.train(); tot=0
        for xb,yb in dl:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE); opt.zero_grad()
            y=lstm(xb); nll=Fnn.mse_loss(y,yb); loss=nll+klw*lstm.kl_loss()
            loss.backward(); opt.step(); tot+=nll.item()*len(xb)
        if (ep+1)%10==0 or ep==0: print(f"  epoch {ep+1:>3}  nll={tot/len(Xt):.4f}")
else:
    lstm=LSTM_MC(X.shape[2], CFG["lstm_hidden"], D, CFG["mc_dropout"]).to(DEVICE)
    opt=torch.optim.Adam(lstm.parameters(), lr=CFG["lr"]); mse=nn.MSELoss()
    for ep in range(CFG["epochs_lstm"]):
        lstm.train(); tot=0
        for xb,yb in dl:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE); opt.zero_grad()
            loss=mse(lstm(xb),yb); loss.backward(); opt.step(); tot+=loss.item()*len(xb)
        if (ep+1)%10==0 or ep==0: print(f"  epoch {ep+1:>3}  mse={tot/len(Xt):.4f}")

# Predicción + incertidumbre sobre todas las ventanas
x_hat, x_var = lstm.predict_unc(Xt.to(DEVICE), n=CFG["mc_samples"])
m_hat = (torch.tensor(x_hat) @ F.T).numpy()          # firmas pronosticadas
m_var = x_var @ (F.numpy().T**2)                      # Var(m)=F^2 Var(x)
ctx   = X[:,-1,D:]                                    # contexto último paso
z = np.concatenate([x_hat, m_hat, ctx], axis=1).astype(np.float32)
print("z (entrada al VAE):", z.shape)

## 6. VAE — detección de anomalías
El VAE se entrena **solo con motores sanos**. Aprende a reconstruir el "estado normal";
un **error de reconstrucción alto** indica anomalía. Entrada: $z=[\hat{x},\hat{m},\text{contexto}]$.

In [ ]:
class VAE(nn.Module):
    def __init__(self, in_dim, lat, hid=64):
        super().__init__()
        self.fc1=nn.Linear(in_dim,hid); self.mu=nn.Linear(hid,lat); self.lv=nn.Linear(hid,lat)
        self.fc2=nn.Linear(lat,hid); self.out=nn.Linear(hid,in_dim)
    def forward(self,x):
        h=torch.relu(self.fc1(x)); mu,lv=self.mu(h),self.lv(h)
        z=mu+torch.exp(0.5*lv)*torch.randn_like(lv)
        return self.out(torch.relu(self.fc2(z))), mu, lv
    @torch.no_grad()
    def recon_error(self,x):
        self.eval(); xh,_,_=self.forward(x); return torch.mean((x-xh)**2,dim=-1)

healthy = (meta["modo_real"]=="Sano").values
if healthy.sum()==0: healthy=np.ones(len(z),dtype=bool)  # datos reales sin etiqueta
z_h = torch.tensor(z[healthy])

vae=VAE(z.shape[1], CFG["vae_latent"], CFG["vae_hidden"]).to(DEVICE)
optv=torch.optim.Adam(vae.parameters(), lr=CFG["lr"])
dlv=DataLoader(TensorDataset(z_h), batch_size=CFG["batch_size"], shuffle=True)
for ep in range(CFG["epochs_vae"]):
    vae.train(); tot=0
    for (zb,) in dlv:
        zb=zb.to(DEVICE); optv.zero_grad()
        zh,mu,lv=vae(zb)
        recon=Fnn.mse_loss(zh,zb); kld=-0.5*torch.mean(1+lv-mu.pow(2)-lv.exp())
        loss=recon+kld; loss.backward(); optv.step(); tot+=loss.item()*len(zb)
    if (ep+1)%15==0 or ep==0: print(f"  epoch {ep+1:>3}  loss={tot/len(z_h):.4f}")
print("VAE entrenado.")

## 7. Score de riesgo multinivel
Por cada muestra:
$$R_i = \alpha\,e + \beta\,|\hat{m}_i| + \gamma\,\mathrm{Var}(\hat{m}_i), \qquad R_{\text{motor}}=\max_i R_i$$
donde $e$ es el error del VAE, $\hat{m}_i$ la activación pronosticada del modo $i$ y $\mathrm{Var}$ su incertidumbre.

**Umbrales auto-calibrados:** $\tau_1,\tau_2,\tau_3$ se fijan con percentiles de la distribución
de riesgo de los **motores sanos** (funciona aun sin etiquetas). Para calibración fina, usa un
motor fallado conocido y ajusta hasta que su Nivel 3 aparezca con antelación.

In [ ]:
# Normalización por referencia "sana" para que los umbrales tengan sentido
e_all = vae.recon_error(torch.tensor(z).to(DEVICE)).cpu().numpy()
e_ref   = np.percentile(e_all[healthy],95)+1e-6
m_ref   = np.percentile(np.abs(m_hat[healthy]),95,axis=0)+1e-6
mvar_ref= np.percentile(m_var[healthy],95,axis=0)+1e-6

def risk(e,m,v):
    e_n=(e/e_ref).reshape(-1,1); m_n=np.abs(m)/m_ref; v_n=v/mvar_ref
    Rm = CFG["alpha"]*e_n + CFG["beta"]*m_n + CFG["gamma"]*v_n
    return Rm, Rm.max(1)

Rm_h,_R_h = risk(e_all[healthy], m_hat[healthy], m_var[healthy])
tau1,tau2,tau3 = (np.percentile(_R_h,90), np.percentile(_R_h,97.5), np.percentile(_R_h,99.5))
print(f"Umbrales auto-calibrados: tau1={tau1:.2f} tau2={tau2:.2f} tau3={tau3:.2f}")

Rm, Rmotor = risk(e_all, m_hat, m_var)
niveles = np.digitize(Rmotor, [tau1,tau2,tau3])
modo_dom = [CFG["failure_modes"][i] for i in Rm.argmax(1)]

out = meta.copy()
out["R_motor"]=Rmotor; out["nivel"]=niveles; out["modo_dominante"]=modo_dom

## 8. Tablero de estado de flota
Última muestra por motor, ordenada por riesgo. Muestra **nivel (0–3)**, **modo dominante** y score.

In [ ]:
estado=(out.sort_values("fecha_muestra").groupby("equipo").tail(1)
            .sort_values("R_motor",ascending=False).reset_index(drop=True))
estado[["equipo","fecha_muestra","nivel","modo_dominante","R_motor","modo_real"]]

## 9. Evolución del riesgo en un motor con falla
Visualiza cómo el score crece muestra a muestra antes de la falla (caso tipo "T3170").

In [ ]:
fallados = out[out["modo_real"].isin(CFG["failure_modes"])]["equipo"].unique()
if len(fallados):
    eid=fallados[0]; s=out[out["equipo"]==eid].sort_values("fecha_muestra")
    plt.figure(figsize=(9,4))
    plt.plot(s["fecha_muestra"], s["R_motor"], marker="o")
    for tau,lbl,col in [(tau1,"τ1","gold"),(tau2,"τ2","orange"),(tau3,"τ3","red")]:
        plt.axhline(tau,ls="--",color=col,label=lbl)
    plt.title(f"Evolución del riesgo — motor {eid} (modo real: {s['modo_real'].iloc[0]})")
    plt.ylabel("R_motor"); plt.xlabel("fecha de muestreo"); plt.legend(); plt.tight_layout()
    plt.show()
else:
    print("No hay motores fallados etiquetados (datos reales sin label de modo).")

## 10. Próximos pasos
1. **Conectar tu Azure SQL** (`USAR_SQL=True`) y validar con datos reales.
2. **Calibrar umbrales** con al menos un caso de falla documentado (que el Nivel 3 aparezca con antelación).
3. **Refinar la matriz F** con criterio tribológico (pesos reales, no solo 0/1).
4. Probar el método **`bbb`** y comparar con **`mc_dropout`** (precisión y calidad de la incertidumbre).
5. Extender a **horizonte largo** (200–500 h) y montar un dashboard de flota.

> **Seguridad:** nunca dejes la contraseña escrita en el notebook ni en archivos que subas a
> GitHub. Usa `getpass` (como aquí) o variables de entorno. Considera **rotar** la credencial en
> Azure y migrar a Azure AD / Managed Identity.